# 06 — Full Dyna-GRPO training

Runs the full method: mixed real/simulated rollouts (gated by predictor uncertainty)
+ counterfactual per-tool credit assignment.

**Hardware**: 2× A100-80G recommended. GPU 0 = actor + ref + LoRA training. GPU 1 = predictor pool.
**Time**: ~24-36 hrs for 500 steps. Set `total_steps` lower for quick runs.


In [ ]:
import sys, os; sys.path.insert(0, str(os.path.abspath(os.path.join(os.getcwd(), '..'))))
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel
from dyna_grpo.config import MODEL, GRPO, DYNA, PATHS
from dyna_grpo.data import numina_math_subset
from dyna_grpo.rewards import reward_for
from dyna_grpo.dyna_grpo import DynaGRPOTrainer
from dyna_grpo.predictor import ToolPredictor, PredictorPool
from dyna_grpo.utils import save_metrics, read_jsonl, set_seed, logger
from dyna_grpo.trace_collector import Trajectory
import json, re
set_seed(GRPO.seed)

RUN_NAME = 'dyna_grpo_full'        # alternatives: 'dyna_no_cf'
USE_CF   = True
ETA      = DYNA.eta

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL.actor_name, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token

actor_base = AutoModelForCausalLM.from_pretrained(
    MODEL.actor_name, torch_dtype=torch.bfloat16, device_map='cuda:0', trust_remote_code=True)
cfg = LoraConfig(r=MODEL.lora_r, lora_alpha=MODEL.lora_alpha,
                  lora_dropout=MODEL.lora_dropout, bias='none',
                  target_modules=list(MODEL.lora_target_modules), task_type='CAUSAL_LM')
actor = get_peft_model(actor_base, cfg)
actor.print_trainable_parameters()
ref = AutoModelForCausalLM.from_pretrained(
    MODEL.actor_name, torch_dtype=torch.bfloat16,
    device_map='cuda:1' if torch.cuda.device_count() > 1 else 'cuda:0',
    trust_remote_code=True).eval()
for p in ref.parameters(): p.requires_grad = False

In [ ]:
# Predictor pool (place all on cuda:1 to keep cuda:0 free for training)
predictor_device = 'cuda:1' if torch.cuda.device_count() > 1 else 'cuda:0'
calib = json.loads((Path(PATHS['logs']) / 'calibration.json').read_text())

predictors, tokenizers, thresholds = {}, {}, {}
for tool in ('calc', 'code', 'search'):
    out_dir = Path(PATHS['ckpts']) / f'predictor_{tool}'
    pt = AutoTokenizer.from_pretrained(out_dir, trust_remote_code=True)
    if pt.pad_token is None: pt.pad_token = pt.eos_token
    pb = AutoModelForCausalLM.from_pretrained(
        MODEL.predictor_base, torch_dtype=torch.bfloat16,
        device_map=predictor_device, trust_remote_code=True)
    pb = PeftModel.from_pretrained(pb, out_dir)
    pred = ToolPredictor(pb, pb.config.hidden_size).to(predictor_device).to(torch.bfloat16)
    aux = torch.load(out_dir / 'aux.pt', map_location=predictor_device)
    pred.unc_head.load_state_dict(aux['unc_head'])
    pred.temperature.data = aux['temperature'].to(predictor_device).to(pred.temperature.dtype)
    pred.eval()
    predictors[tool] = pred; tokenizers[tool] = pt
    thresholds[tool] = float(calib[tool]['threshold'])
pool = PredictorPool(predictors, tokenizers, thresholds)
print('Thresholds:', thresholds)

In [ ]:
ANS_RE = re.compile(r'\\boxed\{(-?\d+)\}')
raw = numina_math_subset(8000)
train_pool = []
for r in raw:
    m = ANS_RE.search((r.get('problem','') or '') + ' ' + (r.get('solution','') or ''))
    if m:
        try: g = int(m.group(1))
        except: continue
        if abs(g) < 10000:
            train_pool.append({'prompt': r['problem'], 'answer': g, 'kind': 'aime'})
    if len(train_pool) >= 500: break
print('Training prompts:', len(train_pool))

In [ ]:
def gen_fn(ctx, max_new):
    enc = tok(ctx, return_tensors='pt', truncation=True, max_length=6000).to('cuda:0')
    with torch.no_grad():
        o = actor.generate(**enc, max_new_tokens=max_new, do_sample=True,
                            temperature=1.0, top_p=0.9,
                            pad_token_id=tok.pad_token_id)
    return tok.decode(o[0][enc.input_ids.size(1):], skip_special_tokens=True)

def reward_fn(traj: Trajectory, sample: dict) -> float:
    return reward_for(sample['kind'], traj.flat_text(), sample)

opt = torch.optim.AdamW([p for p in actor.parameters() if p.requires_grad], lr=GRPO.learning_rate)
trainer = DynaGRPOTrainer(actor, ref, tok, opt, gen_fn, reward_fn, 'cuda:0', GRPO,
                            predictor_pool=pool, lambda_cf=DYNA.lambda_cf, eta=ETA, use_cf=USE_CF)

In [ ]:
import random, time
history = []
log_path = Path(PATHS['logs']) / f'{RUN_NAME}.jsonl'
ckpt_dir = Path(PATHS['ckpts']) / RUN_NAME; ckpt_dir.mkdir(parents=True, exist_ok=True)
wallclock0 = time.time()
random.seed(GRPO.seed)
for step in range(GRPO.total_steps):
    batch = random.sample(train_pool, GRPO.batch_size)
    info = trainer.step([(b['prompt'], b) for b in batch])
    info['step'] = step; info['wallclock_s'] = time.time() - wallclock0
    history.append(info)
    if step % GRPO.log_every == 0:
        print(f'step={step} reward={info["reward_mean"]:.3f} sim_frac={info["sim_call_frac"]:.2f} '
              f'cf={info.get("cf_count",0)} kl={info["kl"]:.4f}')
    if step % GRPO.save_every == 0 and step > 0:
        actor.save_pretrained(ckpt_dir / f'step{step}')
    save_metrics(log_path, {'history': history})
actor.save_pretrained(ckpt_dir / 'final')